In [0]:
from pyspark.sql.functions import *

# Read bronze tables
bank_df = spark.table("workspace.banking_db.bronze_bank")
customer_df = spark.table("workspace.banking_db.bronze_customer")
txn_df = spark.table("workspace.banking_db.bronze_transaction")

# Join ONLY on Customer_ID (common column)
silver_df = txn_df.join(customer_df, "Customer_ID", "left")

# Cleaning and transformations
silver_df = silver_df \
    .filter(col("Transaction_Amount") > 0) \
    .withColumn("transaction_date", to_date(col("Transaction_Date"))) \
    .withColumn("year", year("transaction_date")) \
    .withColumn("month", month("transaction_date")) \
    .withColumn(
        "fraud_flag",
        when(col("Transaction_Amount") > 100000, "HIGH_VALUE")
        .otherwise("NORMAL")
    )

# Save Silver
silver_df.write.mode("overwrite").saveAsTable("workspace.banking_db.silver_transactions")